### **ElGamal Encryption**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/ElGamal_Cipher.ipynb)

> **ElGamal** is a public-key scheme based on the difficulty of the **discrete logarithm** problem. Encryption is **randomized**: the same message gives a different ciphertext every time. Textbook ElGamal is also **malleable**: anyone can multiply the hidden message by a known value without decrypting it.

> [!WARNING]
> Educational code. Never use these implementations to protect real data: use a vetted library (for example, `cryptography`) with modern, authenticated algorithms.

#### **01. Keys, Encryption, and Decryption**

* Public parameters: a large prime `p` and a generator `g`.
* Private key `x`; public key `y = g^x mod p`.
* Encrypt `m` with a fresh random `k`: `c1 = g^k`, `c2 = m · y^k` (mod p).
* Decrypt: `m = c2 · (c1^x)^(-1) mod p`.

The demo uses a 64-bit **safe prime** so the numbers stay readable; real systems need at least 2048-bit groups (or elliptic curves). Randomness comes from `secrets`, the cryptographically secure generator, not `random`.

In [1]:
# Safe-prime group and key generation (sizes for demonstration only)
import secrets

def is_probable_prime(n, rounds=40):
    """Miller-Rabin primality test."""
    if n < 2:
        return False
    for small in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29):
        if n % small == 0:
            return n == small
    d, s = n - 1, 0
    while d % 2 == 0:
        d, s = d // 2, s + 1
    for _ in range(rounds):
        x = pow(secrets.randbelow(n - 3) + 2, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def safe_prime(bits):
    """Returns p = 2q + 1 with q prime; its group order has only the factors 2 and q."""
    while True:
        q = secrets.randbits(bits - 1) | (1 << (bits - 2)) | 1
        if is_probable_prime(q) and is_probable_prime(2 * q + 1):
            return 2 * q + 1

def generator(p):
    q = (p - 1) // 2
    while True:
        g = secrets.randbelow(p - 3) + 2
        if pow(g, 2, p) != 1 and pow(g, q, p) != 1:
            return g

p = safe_prime(64)
g = generator(p)
x = secrets.randbelow(p - 3) + 2          # private key
y = pow(g, x, p)                          # public key
print(f"p = {p}\ng = {g}\npublic y = {y}\nprivate x = {x}")

p = 9901892757099920099
g = 7039857368216491033
public y = 6189792909298851513
private x = 7144922417505721591


In [2]:
# Encrypt and decrypt; the same message gives a different ciphertext each time
def encrypt(m, public):
    p, g, y = public
    k = secrets.randbelow(p - 3) + 2      # fresh ephemeral key for every message
    return pow(g, k, p), (m * pow(y, k, p)) % p

def decrypt(ciphertext, x, p):
    c1, c2 = ciphertext
    return (c2 * pow(c1, -x, p)) % p      # pow with a negative exponent = modular inverse

public = (p, g, y)
message = int.from_bytes(b"secret", "big")          # text encoded as an integer < p
for attempt in (1, 2):
    ct = encrypt(message, public)
    print(f"Ciphertext {attempt}: {ct}")
recovered = decrypt(ct, x, p)
print("Decrypted:", recovered.to_bytes((recovered.bit_length() + 7) // 8, "big"))
assert recovered == message

Ciphertext 1: (7059707408332794939, 9557793460040885438)
Ciphertext 2: (2090427320191695732, 3729946753729974011)
Decrypted: b'secret'


#### **02. Malleability**

Multiplying `c2` by `t` multiplies the hidden message by `t`. An attacker who intercepts an encrypted amount can double it without knowing the key. Real protocols prevent this with padding schemes and authentication (for example, ECIES or hybrid encryption with AES-GCM).

In [3]:
# Double an encrypted amount without the private key
amount = 250
c1, c2 = encrypt(amount, public)
tampered = (c1, (c2 * 2) % p)
print("Original amount :", decrypt((c1, c2), x, p))
print("Tampered amount :", decrypt(tampered, x, p))
assert decrypt(tampered, x, p) == 2 * amount

Original amount : 250
Tampered amount : 500


#### **Key Takeaways**

* Security rests on the discrete logarithm problem in a large, well-chosen group.
* Use a fresh random `k` for every message; reusing it reveals the ratio between plaintexts.
* Textbook ElGamal is malleable; use it only inside a vetted scheme.